# 06 — Leakage Experiment: Adding `duration` Back In

This notebook runs a controlled experiment: take the exact same Logistic Regression and Random Forest setups from `05_model_development.ipynb` (same features, same preprocessing strategy, same model configurations, same random seeds, same saved split indices), and add back the one feature the whole audit excluded — **`duration`** — to measure exactly how much it inflates performance.

If `duration` behaves the way the Phase-1 EDA and Phase-2 audit predicted (a near-outcome proxy only knowable after the call ends), we should see a **large, suspicious jump** in every metric — not a modest improvement. That jump *is* the leakage, made visible and quantified.

As in Phase 4/5: preprocessing and models are fit **only** on training data and evaluated **only** on validation data for each split strategy; the test set is never touched; nothing from the earlier phases (saved splits, saved no-duration models, the dataset file) is modified.

In [ ]:
import os

import numpy as np
import pandas as pd
from joblib import dump

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
)

RANDOM_STATE = 42
RESULTS_DIR = "../results"
MODELS_DIR = f"{RESULTS_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

pd.set_option("display.max_columns", 100)

## Load the dataset, existing split indices, and the Phase-5 (no-duration) results (read-only)

All three are read, never modified: the raw dataset, `results/split_stratified.csv` / `results/split_chronological.csv`, and `results/model_comparison.csv` (the no-duration baseline we'll compare against).

In [ ]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df_raw = pd.read_csv(DATA_PATH, sep=";")

split_paths = {
    "stratified": f"{RESULTS_DIR}/split_stratified.csv",
    "chronological": f"{RESULTS_DIR}/split_chronological.csv",
}
no_duration_results_path = f"{RESULTS_DIR}/model_comparison.csv"
for path in list(split_paths.values()) + [no_duration_results_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run 03_preprocessing.ipynb and 05_model_development.ipynb first.")

splits = {name: pd.read_csv(path) for name, path in split_paths.items()}
no_duration_results = pd.read_csv(no_duration_results_path)
no_duration_results = no_duration_results[no_duration_results["model"] != "majority_baseline"].copy()

print(f"Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
for name, table in splits.items():
    print(f"{name} split sizes:", table["split"].value_counts().to_dict())
no_duration_results

## Build the working dataset: 14 realistic features + `duration`

Identical to the Phase-3/Phase-5 feature selection and `pdays` recoding, with one addition: `duration` is now included as a 15th (numerical) feature. Everything else — the other 13 raw columns, the `pdays` → `previously_contacted` / `days_since_previous_contact` split, `"unknown"` handling — is unchanged.

In [ ]:
PRIMARY_FEATURES = [
    "age", "job", "marital", "education", "default", "housing", "loan",
    "pdays", "previous", "poutcome",
    "contact", "month", "day_of_week",
]
TARGET = "y"
assert len(PRIMARY_FEATURES) == 13

data = df_raw[PRIMARY_FEATURES + ["duration"] + [TARGET]].copy()
data["previously_contacted"] = (data["pdays"] != 999).astype(int)
data["days_since_previous_contact"] = data["pdays"].replace(999, np.nan)
data = data.drop(columns=["pdays"])

numerical_features = ["age", "previous", "days_since_previous_contact", "previously_contacted", "duration"]
categorical_features = ["job", "marital", "education", "default", "housing", "loan",
                         "poutcome", "contact", "month", "day_of_week"]
FEATURE_COLUMNS = numerical_features + categorical_features

assert len(FEATURE_COLUMNS) == 15
assert "duration" in FEATURE_COLUMNS

print(f"Working dataset shape: {data.shape} ({len(FEATURE_COLUMNS)} features + target, including duration)")
data.head()

## Preprocessing + model pipelines (identical config to Phase 5)

Same imputation/encoding strategy, same `StandardScaler`-only-for-LR rule, same model hyperparameters and `random_state=42`. `duration` simply joins the numerical feature list — no special treatment, since the entire point is to see what happens when it's treated like any other feature.

- **Logistic Regression**: `max_iter=1000`, `random_state=42` (unchanged from Phase 5).
- **Random Forest**: `n_estimators=100`, `random_state=42`, `n_jobs=-1` (unchanged from Phase 5).

In [ ]:
def make_categorical_pipeline():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])


def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(transformers=[
        ("num", Pipeline(steps=numeric_steps), numerical_features),
        ("cat", make_categorical_pipeline(), categorical_features),
    ])


def make_model_pipelines():
    """Fresh, unfit pipelines for one (split) experiment — never reused across splits."""
    return {
        "logistic_regression": Pipeline(steps=[
            ("preprocess", make_preprocessor(scale_numeric=True)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "random_forest": Pipeline(steps=[
            ("preprocess", make_preprocessor(scale_numeric=False)),
            ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
    }


make_model_pipelines()

## Run the experiments: 2 split strategies × 2 models, with `duration` included

Same evaluation protocol as Phase 5: fit on training rows only, predict on validation rows only, test set untouched.

In [ ]:
POS_LABEL = "yes"


def positive_class_proba(fitted_pipeline, X):
    pos_idx = list(fitted_pipeline.classes_).index(POS_LABEL)
    return fitted_pipeline.predict_proba(X)[:, pos_idx]


def run_experiment(split_name, split_table, model_name, pipeline):
    train_idx = split_table.loc[split_table["split"] == "train", "row_index"]
    val_idx = split_table.loc[split_table["split"] == "val", "row_index"]

    X_train = data.loc[train_idx, FEATURE_COLUMNS]
    y_train = data.loc[train_idx, TARGET]
    X_val = data.loc[val_idx, FEATURE_COLUMNS]
    y_val = data.loc[val_idx, TARGET]

    # Fit ONLY on training data.
    pipeline.fit(X_train, y_train)

    # Evaluate ONLY on validation data (test set is never touched here).
    y_pred = pipeline.predict(X_val)
    y_proba = positive_class_proba(pipeline, X_val)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=["no", "yes"]).ravel()

    metrics = {
        "split_strategy": split_name,
        "model": model_name,
        "feature_set": "with_duration",
        "val_prevalence": float((y_val == POS_LABEL).mean()),
        "average_precision": average_precision_score((y_val == POS_LABEL).astype(int), y_proba),
        "precision": precision_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "recall": recall_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1": f1_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "roc_auc": roc_auc_score((y_val == POS_LABEL).astype(int), y_proba),
        "accuracy": accuracy_score(y_val, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

    predictions = pd.DataFrame({
        "row_index": val_idx.values,
        "split_strategy": split_name,
        "model": model_name,
        "feature_set": "with_duration",
        "y_true": y_val.values,
        "y_pred": y_pred,
        "y_proba_yes": y_proba,
    })

    return pipeline, metrics, predictions


fitted_pipelines = {}
with_duration_metrics = []
with_duration_predictions = []

for split_name, split_table in splits.items():
    pipelines_for_split = make_model_pipelines()
    for model_name, pipeline in pipelines_for_split.items():
        fitted_pipeline, metrics, predictions = run_experiment(split_name, split_table, model_name, pipeline)
        fitted_pipelines[(split_name, model_name)] = fitted_pipeline
        with_duration_metrics.append(metrics)
        with_duration_predictions.append(predictions)
        print(f"[{split_name:13s} | {model_name:20s}] "
              f"AP={metrics['average_precision']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}  "
              f"F1={metrics['f1']:.4f}  prevalence={metrics['val_prevalence']:.4f}")

with_duration_df = pd.DataFrame(with_duration_metrics)
with_duration_predictions_df = pd.concat(with_duration_predictions, ignore_index=True)

## Compare with the no-duration (Phase 5) results

For each (split strategy, model), we join the no-duration metrics against the with-duration metrics and compute the deltas: **ΔAP**, ΔROC-AUC, ΔPrecision, ΔRecall, ΔF1 (all `with_duration − no_duration`). A large positive ΔAP/ΔROC-AUC driven by a single feature is exactly the leakage signature we expect to see here.

In [ ]:
no_duration_compare = no_duration_results[[
    "split_strategy", "model", "val_prevalence",
    "average_precision", "roc_auc", "precision", "recall", "f1",
]].rename(columns={
    "val_prevalence": "val_prevalence_no_duration",
    "average_precision": "average_precision_no_duration",
    "roc_auc": "roc_auc_no_duration",
    "precision": "precision_no_duration",
    "recall": "recall_no_duration",
    "f1": "f1_no_duration",
})

with_duration_compare = with_duration_df[[
    "split_strategy", "model", "val_prevalence",
    "average_precision", "roc_auc", "precision", "recall", "f1",
]].rename(columns={
    "val_prevalence": "val_prevalence_with_duration",
    "average_precision": "average_precision_with_duration",
    "roc_auc": "roc_auc_with_duration",
    "precision": "precision_with_duration",
    "recall": "recall_with_duration",
    "f1": "f1_with_duration",
})

leakage_comparison = no_duration_compare.merge(with_duration_compare, on=["split_strategy", "model"], how="inner")

assert len(leakage_comparison) == 4, "Expected exactly 4 rows (2 splits x 2 models)."
assert np.allclose(
    leakage_comparison["val_prevalence_no_duration"], leakage_comparison["val_prevalence_with_duration"]
), "Validation prevalence should be identical — same split, same rows, only features changed."

leakage_comparison["delta_average_precision"] = (
    leakage_comparison["average_precision_with_duration"] - leakage_comparison["average_precision_no_duration"]
)
leakage_comparison["delta_roc_auc"] = (
    leakage_comparison["roc_auc_with_duration"] - leakage_comparison["roc_auc_no_duration"]
)
leakage_comparison["delta_precision"] = (
    leakage_comparison["precision_with_duration"] - leakage_comparison["precision_no_duration"]
)
leakage_comparison["delta_recall"] = (
    leakage_comparison["recall_with_duration"] - leakage_comparison["recall_no_duration"]
)
leakage_comparison["delta_f1"] = (
    leakage_comparison["f1_with_duration"] - leakage_comparison["f1_no_duration"]
)

leakage_comparison = leakage_comparison.sort_values(["split_strategy", "model"]).reset_index(drop=True)
leakage_comparison

## Save predicted probabilities, fitted models, and the leakage comparison CSV

New files only — nothing from Phase 5 is overwritten:

- `results/leakage_predictions.csv` — validation predictions/probabilities for the with-duration models (`feature_set="with_duration"`, to distinguish from `validation_predictions.csv`'s no-duration rows).
- `results/models/<split_strategy>__<model>__with_duration.joblib` — the 4 newly fitted pipelines, under distinct filenames from the Phase-5 models.
- `results/leakage_comparison.csv` — the no-duration vs. with-duration metrics and deltas computed above.

In [ ]:
predictions_path = f"{RESULTS_DIR}/leakage_predictions.csv"
comparison_path = f"{RESULTS_DIR}/leakage_comparison.csv"

with_duration_predictions_df.to_csv(predictions_path, index=False)
leakage_comparison.to_csv(comparison_path, index=False)

existing_model_files = set(os.listdir(MODELS_DIR)) if os.path.isdir(MODELS_DIR) else set()

for (split_name, model_name), fitted_pipeline in fitted_pipelines.items():
    model_filename = f"{split_name}__{model_name}__with_duration.joblib"
    assert model_filename not in existing_model_files, \
        f"Refusing to overwrite unexpected existing file: {model_filename}"
    dump(fitted_pipeline, f"{MODELS_DIR}/{model_filename}")

print("Saved:", predictions_path, "-", with_duration_predictions_df.shape)
print("Saved:", comparison_path, "-", leakage_comparison.shape)
print(f"Saved {len(fitted_pipelines)} new fitted pipelines (with_duration) to {MODELS_DIR}/")

## Summary

- Repeated the exact Phase-5 Logistic Regression / Random Forest setup (same other features, preprocessing, hyperparameters, seeds, splits) with `duration` added as a 15th feature.
- Fit only on training data, evaluated only on validation data, for both the stratified and chronological splits; test set untouched throughout.
- Computed ΔAP, ΔROC-AUC, ΔPrecision, ΔRecall, ΔF1 against the Phase-5 no-duration results for each (split, model) pair.
- Saved `results/leakage_predictions.csv`, `results/leakage_comparison.csv`, and 4 newly fitted pipelines under `results/models/*__with_duration.joblib` — none of the Phase-5 artifacts (models, splits, dataset) were modified or overwritten.

See the comparison table above for the actual magnitude of the jump; a large, consistent increase in AP/ROC-AUC attributable to a single feature known only after the call ends is the leakage signature this whole project has been building toward demonstrating.